# 配套实践 16-01：用 CEM 优化二维动作序列

本练习让二维点机器人从起点绕开圆形障碍到达目标。我们先把目标、碰撞、动作幅度和平滑性写成可分别检查的代价，再用 Cross-Entropy Method（CEM）迭代更新动作序列分布。重点不是搭建复杂仿真，而是看清“采样—预测—评价—elite 更新”的完整数据流。依赖：NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/16-objective-cem-and-mpc/" target="_blank">在新标签页返回课程正文</a>


In [ ]:
import numpy as np  # 生成候选动作并进行向量化轨迹计算
import matplotlib.pyplot as plt  # 绘制任务、搜索收敛与候选轨迹
from matplotlib.patches import Circle  # 在二维平面中显示圆形障碍
np.random.seed(16)  # 固定 NumPy 全局随机种子以便复现实验
plt.rcParams["figure.dpi"] = 120  # 提高笔记本内图像的显示清晰度
print("本练习使用解析二维 World Model，不需要 GPU。")  # 告知读者当前实践的运行要求


## 1. 先明确动力学与评价项

状态是二维位置 $p_t=(x_t,y_t)$，动作是每一步位移 $a_t$，教学用 World Model 为 $p_{t+1}=p_t+a_t$。总成本由终点目标误差、碰撞、障碍附近软代价、动作幅度和平滑性组成。硬碰撞罚项远大于其他项，但最终发送动作前仍应有独立安全检查。


In [ ]:
start = np.array([-1.10, -0.75])  # 设置点机器人的起始位置
goal = np.array([1.10, 0.80])  # 设置希望到达的目标位置
obstacle_center = np.array([0.0, 0.0])  # 设置圆形障碍中心
obstacle_radius = 0.38  # 设置不可进入的障碍半径
horizon = 22  # 设置规划动作序列的长度
action_limit = 0.16  # 限制每个坐标单步可移动的最大距离
def rollout_sequences(action_sequences):  # 定义批量解析 World Model
    cumulative_motion = np.cumsum(action_sequences, axis=1)  # 沿时间累计每条候选的动作位移
    predicted_positions = start[None, None, :] + cumulative_motion  # 从共同起点得到全部候选未来位置
    return predicted_positions  # 返回形状为候选数乘 horizon 乘二维的位置数组
def evaluate_sequences(action_sequences):  # 定义可分解的候选未来评价器
    predicted_positions = rollout_sequences(action_sequences)  # 用 World Model 批量预测候选轨迹
    obstacle_distances = np.linalg.norm(predicted_positions - obstacle_center, axis=2)  # 计算每个未来位置到障碍中心的距离
    goal_cost = 25.0 * np.sum((predicted_positions[:, -1] - goal) ** 2, axis=1)  # 惩罚 horizon 末端与目标的平方距离
    collision_cost = 450.0 * np.sum(obstacle_distances < obstacle_radius, axis=1)  # 对每个进入障碍的时刻施加大额代价
    clearance_cost = 15.0 * np.sum(np.maximum(obstacle_radius + 0.12 - obstacle_distances, 0.0) ** 2, axis=1)  # 在障碍外侧建立连续缓冲代价
    effort_cost = 0.15 * np.sum(action_sequences ** 2, axis=(1, 2))  # 轻微惩罚过大的动作幅度
    smoothness_cost = 0.30 * np.sum(np.diff(action_sequences, axis=1) ** 2, axis=(1, 2))  # 惩罚相邻动作的剧烈变化
    total_cost = goal_cost + collision_cost + clearance_cost + effort_cost + smoothness_cost  # 汇总用于候选排序的总代价
    components = {"goal": goal_cost, "collision": collision_cost, "clearance": clearance_cost, "effort": effort_cost, "smoothness": smoothness_cost}  # 保留所有分项以支持诊断
    return total_cost, components  # 同时返回总代价和可解释分项
straight_velocity = (goal - start) / horizon  # 构造每一步完全相同的直线动作
straight_actions = np.tile(straight_velocity, (horizon, 1))[None, :, :]  # 将直线动作整理为单候选批次
straight_path = np.vstack([start, rollout_sequences(straight_actions)[0]])  # 把起点加入直线预测轨迹
straight_total, straight_components = evaluate_sequences(straight_actions)  # 评价直接通向目标的朴素计划
figure, ax = plt.subplots(figsize=(7.2, 5.0), constrained_layout=True)  # 建立初始任务示意图
ax.add_patch(Circle(obstacle_center, obstacle_radius, color="#d66b5d", alpha=0.35, label="Obstacle"))  # 绘制不可进入障碍区域
ax.add_patch(Circle(obstacle_center, obstacle_radius + 0.12, fill=False, edgecolor="#d66b5d", linestyle="--", linewidth=1.5, label="Soft clearance"))  # 绘制连续缓冲区域边界
ax.plot(straight_path[:, 0], straight_path[:, 1], color="#73859b", linewidth=2.0, marker=".", label="Straight plan")  # 显示未经搜索的直线计划
ax.scatter(*start, s=85, color="#245b78", zorder=5, label="Start")  # 标出任务起点
ax.scatter(*goal, s=130, marker="*", color="#2f855a", zorder=5, label="Goal")  # 标出任务目标
ax.set(xlabel="x position", ylabel="y position", title="The shortest straight plan is not feasible", xlim=(-1.4, 1.4), ylim=(-1.15, 1.2), aspect="equal")  # 设置坐标范围和等比例显示
ax.legend(fontsize=8, loc="upper left")  # 显示场景元素图例
ax.grid(alpha=0.18)  # 添加浅色网格帮助读取位置
plt.show()  # 在笔记本中输出初始任务图
print(f"直线计划总代价：{straight_total[0]:.1f}")  # 报告直线计划的总代价
print(f"其中碰撞代价：{straight_components['collision'][0]:.1f}，目标代价：{straight_components['goal'][0]:.3f}")  # 显示高总分真正来自碰撞而非目标误差


**怎样理解结果：** 直线路径最终准确到达目标，所以目标代价接近零；它却穿过障碍，碰撞分项使总代价非常高。这说明“离目标近”不能代替完整任务评价，也说明分项日志比单一总分更容易定位问题。


## 2. 用 elite 反复更新采样分布

CEM 为每个时间位置维护二维动作均值和标准差。初始均值大致指向目标，标准差保留绕行空间；每轮采样 500 条序列，选择成本最低的 50 条作为 elite。更新时保留少量旧统计，并设置最小标准差，避免搜索过早坍缩。


In [ ]:
random_generator = np.random.default_rng(16)  # 为 CEM 建立独立且可复现的随机数生成器
sample_count = 500  # 设置每轮并行评价的候选序列数量
elite_count = 50  # 设置用于更新分布的低代价候选数量
iteration_count = 7  # 设置 CEM 采样和更新轮数
initial_direction = goal - start  # 计算从起点指向目标的方向
initial_action = initial_direction / np.linalg.norm(initial_direction) * 0.12  # 将方向缩放到动作边界内部
action_mean = np.tile(initial_action, (horizon, 1))  # 用简单 Action Model proposal 初始化各时刻均值
action_std = np.full((horizon, 2), 0.10)  # 使用较宽初始标准差保留不同绕行路线
best_so_far = None  # 保存跨迭代找到的全局最佳动作序列
best_so_far_cost = np.inf  # 用无穷大初始化全局最佳代价
best_cost_history = []  # 记录每轮结束后的全局最佳代价
elite_cost_history = []  # 记录每轮 elite 平均代价
distribution_width_history = []  # 记录采样分布的平均标准差
candidate_snapshots = []  # 保存若干轮候选用于后续可视化
for iteration_index in range(iteration_count):  # 重复执行采样、评价、筛选和更新
    sampled_noise = random_generator.standard_normal((sample_count, horizon, 2))  # 为当前分布采样标准高斯噪声
    action_sequences = action_mean[None, :, :] + action_std[None, :, :] * sampled_noise  # 按当前均值和标准差生成候选动作
    action_sequences = np.clip(action_sequences, -action_limit, action_limit)  # 把所有动作裁剪到执行器边界
    sequence_costs, unused_components = evaluate_sequences(action_sequences)  # 通过 World Model 和评价器计算每条候选代价
    elite_indices = np.argsort(sequence_costs)[:elite_count]  # 找出代价最低的 elite 候选索引
    elite_sequences = action_sequences[elite_indices]  # 取出用于拟合下一轮分布的动作序列
    current_best_index = int(np.argmin(sequence_costs))  # 找出当前轮最低代价候选
    if sequence_costs[current_best_index] < best_so_far_cost:  # 仅在当前候选确实更好时更新全局结果
        best_so_far_cost = float(sequence_costs[current_best_index])  # 保存新的全局最低代价
        best_so_far = action_sequences[current_best_index].copy()  # 保存新的全局最佳动作序列
    candidate_snapshots.append((action_sequences.copy(), sequence_costs.copy(), elite_indices.copy()))  # 保存当前候选和 elite 供图形解释
    new_mean = elite_sequences.mean(axis=0)  # 用 elite 在每个 horizon 的均值拟合新分布中心
    new_std = elite_sequences.std(axis=0)  # 用 elite 在每个 horizon 的标准差拟合新分布宽度
    action_mean = 0.20 * action_mean + 0.80 * new_mean  # 平滑更新均值以减少单轮随机波动
    action_std = np.maximum(0.012, 0.20 * action_std + 0.80 * new_std)  # 平滑更新标准差并保留最小探索宽度
    best_cost_history.append(best_so_far_cost)  # 记录当前为止找到的最佳代价
    elite_cost_history.append(float(sequence_costs[elite_indices].mean()))  # 记录当前 elite 的平均质量
    distribution_width_history.append(float(action_std.mean()))  # 记录当前采样分布平均宽度
figure, axes = plt.subplots(1, 2, figsize=(10.8, 3.9), constrained_layout=True)  # 建立成本变化和分布宽度两个子图
iteration_axis = np.arange(1, iteration_count + 1)  # 建立从一开始的迭代编号横轴
axes[0].plot(iteration_axis, best_cost_history, marker="o", linewidth=2.1, color="#2f855a", label="Best so far")  # 绘制全局最佳代价的单调改善
axes[0].plot(iteration_axis, elite_cost_history, marker="s", linewidth=1.8, color="#d17a3a", label="Elite mean")  # 绘制 elite 群体的平均代价
axes[0].set(xlabel="CEM iteration", ylabel="Cost", title="Low-cost candidates improve")  # 标注候选质量变化图
axes[0].legend(fontsize=8)  # 显示两条成本曲线的图例
axes[0].grid(alpha=0.2)  # 添加浅色网格帮助比较迭代
axes[1].plot(iteration_axis, distribution_width_history, marker="o", linewidth=2.1, color="#586f9b")  # 绘制动作分布逐轮收窄过程
axes[1].set(xlabel="CEM iteration", ylabel="Mean action standard deviation", title="Sampling distribution becomes focused")  # 标注分布宽度图
axes[1].grid(alpha=0.2)  # 添加浅色网格帮助读取宽度
plt.show()  # 在笔记本中输出 CEM 收敛诊断


**怎样理解结果：** best-so-far 只在找到更好候选时下降；elite 平均值衡量当前低代价群体的整体质量；动作标准差下降表示计算预算逐渐集中。标准差越小并不总越好，若很早降到接近零，搜索可能只保留障碍一侧的局部路线。


## 3. 观察候选轨迹怎样从发散变为聚焦

下面分别显示第 1、4、7 轮中成本最低的 25 条轨迹，并单独显示整个搜索期间的最佳计划。浅线不是机器人真实执行历史，而是 World Model 在一次规划调用中想象的候选未来。


In [ ]:
selected_iterations = [0, 3, 6]  # 选择搜索早期、中期和末期三个快照
figure, axes = plt.subplots(1, 3, figsize=(13.0, 4.0), constrained_layout=True)  # 建立三个候选轨迹快照子图
for plot_index, iteration_index in enumerate(selected_iterations):  # 逐个绘制选中的 CEM 迭代
    action_sequences, sequence_costs, elite_indices = candidate_snapshots[iteration_index]  # 读取当前迭代的候选和 elite
    elite_paths = rollout_sequences(action_sequences[elite_indices[:25]])  # 预测当前最低代价二十五条轨迹
    axes[plot_index].add_patch(Circle(obstacle_center, obstacle_radius, color="#d66b5d", alpha=0.35))  # 在当前子图绘制障碍区域
    for elite_path in elite_paths:  # 逐条显示当前迭代的低代价候选
        complete_path = np.vstack([start, elite_path])  # 为预测轨迹补上共同起点
        axes[plot_index].plot(complete_path[:, 0], complete_path[:, 1], color="#7e9bb7", linewidth=0.9, alpha=0.35)  # 用浅线显示候选未来
    lowest_index = elite_indices[0]  # 找出当前轮代价最低的候选
    lowest_path = np.vstack([start, rollout_sequences(action_sequences[lowest_index:lowest_index + 1])[0]])  # 得到当前轮最佳完整路径
    axes[plot_index].plot(lowest_path[:, 0], lowest_path[:, 1], color="#245b78", linewidth=2.2, label="Iteration best")  # 突出当前轮最佳路径
    axes[plot_index].scatter(*start, s=55, color="#245b78", zorder=5)  # 标出每幅图的共同起点
    axes[plot_index].scatter(*goal, s=90, marker="*", color="#2f855a", zorder=5)  # 标出每幅图的共同目标
    axes[plot_index].set(title=f"Iteration {iteration_index + 1}", xlim=(-1.4, 1.4), ylim=(-1.15, 1.2), aspect="equal")  # 使用相同范围便于跨迭代比较
    axes[plot_index].grid(alpha=0.16)  # 添加浅色网格帮助观察候选聚焦
plt.show()  # 在笔记本中输出候选轨迹演化图
best_path = np.vstack([start, rollout_sequences(best_so_far[None, :, :])[0]])  # 用全局最佳动作得到完整预测路径
best_total, best_components = evaluate_sequences(best_so_far[None, :, :])  # 重新计算最佳路径的所有代价分量
best_clearance = np.min(np.linalg.norm(best_path - obstacle_center, axis=1))  # 计算最佳路径到障碍中心的最小距离
final_goal_error = np.linalg.norm(best_path[-1] - goal)  # 计算最佳路径末端的目标误差
figure, ax = plt.subplots(figsize=(7.2, 5.0), constrained_layout=True)  # 建立最终计划诊断图
ax.add_patch(Circle(obstacle_center, obstacle_radius, color="#d66b5d", alpha=0.35, label="Obstacle"))  # 绘制最终计划必须避开的障碍
ax.add_patch(Circle(obstacle_center, obstacle_radius + 0.12, fill=False, edgecolor="#d66b5d", linestyle="--", linewidth=1.4, label="Soft clearance"))  # 绘制障碍缓冲区边界
ax.plot(best_path[:, 0], best_path[:, 1], color="#245b78", linewidth=2.5, marker=".", label="CEM best plan")  # 显示 CEM 找到的全局最佳路径
ax.scatter(*start, s=85, color="#245b78", zorder=5, label="Start")  # 标出最终图的起点
ax.scatter(*goal, s=130, marker="*", color="#2f855a", zorder=5, label="Goal")  # 标出最终图的目标
ax.set(xlabel="x position", ylabel="y position", title="CEM finds a feasible action sequence", xlim=(-1.4, 1.4), ylim=(-1.15, 1.2), aspect="equal")  # 设置最终路径图的坐标和标题
ax.legend(fontsize=8, loc="upper left")  # 显示最终路径图例
ax.grid(alpha=0.18)  # 添加浅色网格帮助读取最终位置
plt.show()  # 在笔记本中输出最终 CEM 计划
print(f"最佳计划总代价：{best_total[0]:.3f}")  # 报告最佳候选的总代价
print(f"目标误差：{final_goal_error:.3f}；最小中心距：{best_clearance:.3f}；障碍半径：{obstacle_radius:.3f}")  # 联合报告到达精度与安全余量
print(f"分项代价：目标 {best_components['goal'][0]:.3f}，碰撞 {best_components['collision'][0]:.1f}，缓冲 {best_components['clearance'][0]:.3f}，动作 {best_components['effort'][0]:.3f}，平滑 {best_components['smoothness'][0]:.3f}")  # 输出完整分项以检查优化取舍


**怎样理解结果：** 早期 elite 仍有多种形状，后期逐渐聚集到障碍同一侧。最终路径的最小中心距应大于障碍半径，碰撞分项为零，同时末端接近目标。若只看总代价，无法确认低分来自真正安全到达，还是某个权重设置过小；因此最后仍要检查几何余量和各分项。

**本练习的结论：** CEM 不直接学习策略，而是在当前状态上调用 World Model 和评价器搜索动作序列。它能处理不可微的碰撞事件，也需要足够样本、合理初始化和防坍缩设置。这里得到的仍是一段模型内计划；下一份实践将加入真实扰动，并比较一次性执行与每步重新规划。
